# 06 · Power BI export

Power BI reads the star schema from CSV files in `powerbi/data/`. After every new collection: run this notebook (or `python run_pipeline.py`) and press **Refresh** in Power BI.

The step-by-step report build — relationships, measures and the six pages — is in `powerbi/POWERBI_BUILD_GUIDE.md` and `powerbi/DAX_MEASURES.md`.

In [1]:
import sys, sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 45)

import export_powerbi
out = export_powerbi.export()

pd.DataFrame([{"file": p.name, "rows": sum(1 for _ in open(p, encoding="utf-8-sig")) - 1,
               "size_kb": round(p.stat().st_size / 1024, 1)}
              for p in sorted(out.glob("*.csv"))])

,file,rows,size_kb
0,bridge_commodity_cpi.csv,39,0.5
1,dim_brand.csv,234,3.8
2,dim_category.csv,26,0.9
3,dim_cpi_item.csv,383,17.1
4,dim_date.csv,1461,85.8
5,dim_location.csv,204,24.0
6,dim_product.csv,419,50.1
7,dim_source.csv,3,1.5
8,dq_check_summary.csv,54,5.7
9,dq_issue_log.csv,603,95.1


## Report pages

| Page | Built from |
|---|---|
| 1 · Consumer Price Overview | KPI cards, top YoY risers, category trend lines (DoCA) |
| 2 · Product Price Trends | commodity slicer, price line, MoM / YoY table, optional CPI benchmark line |
| 3 · Brand Comparison | category + unit slicers (single select), brand ₹/kg or ₹/l bars with sample flag |
| 4 · Location Comparison | state map, city index, same-product spread, retail vs wholesale |
| 5 · Discount & Availability | discount KPIs, list vs effective price, DoCA reporting-status matrix |
| 6 · Data Quality | check counts, issue log, raw-file provenance (no analysis filter on this page) |

Pages 1–5 use a report-level filter `fact_price[include_in_analysis] = 1`; page 6 shows everything.

## Quick check that the export matches the database

In [2]:
from config import DB_PATH
con = sqlite3.connect(DB_PATH)
db_rows = con.execute("SELECT COUNT(*) FROM fact_price").fetchone()[0]
csv_rows = len(pd.read_csv(out / "fact_price.csv"))
print(f"fact_price: database {db_rows} rows, CSV {csv_rows} rows ->", "OK" if db_rows == csv_rows else "MISMATCH")
con.close()

fact_price: database 677 rows, CSV 677 rows -> OK
